# Phase 5: Circular DDD Handcrafted Feature Engine
### 256-Dimensional Directional Distance Distribution & Structural Geometry
---
### What is Phase 5 and why do we need it?
In deep learning, people often rely on huge multi-layer convolutional networks to discover features automatically from millions of images. But **our specification strictly prohibits deep learning**.

Instead, we use a time-tested classical pattern recognition technique: **Handcrafted Structural & Topological Feature Extraction**.

Rather than treating a character as just an array of raw pixel values, we measure **physical geometric properties**:
1. **Where are the open cavities?** (e.g., the right-facing opening of 'C', or the top opening of 'U')
2. **Where are the closed loops/holes?** (e.g., the hollow center of 'O', '0', '8', 'B', 'P')
3. **How thick are the pen strokes?** (e.g., crossing junctions in 'X' or '+' vs single lines in 'I')
4. **Where is the ink mass concentrated?** (top-heavy in 'T' or 'P' vs bottom-heavy in 'L' or '_')

**How Circular DDD Works (Wong & Sebastian 2015, Katiyar & Mehfuz 2015):**
- We inscribe the $16 \times 16$ character inside a virtual circle of radius $R = 7.5$.
- We place anchor points along **8 concentric radial rings**.
- From each anchor, we cast rays along **8 compass directions** (East, NE, North, NW, West, SW, South, SE).
- Along each ray, we calculate **4 Dual-Set metrics**:
  * **W-Set:** $W_{\text{dist}}$ (distance to stroke) and $W_{\text{span}}$ (interior loop space).
  * **B-Set:** $B_{\text{thick}}$ (stroke thickness) and $B_{\text{dens}}$ (ink mass density).
- Total features: **8 rings × 8 directions × 4 metrics = exactly 256 numbers**!

In [ ]:
# Step 1: Import required libraries and Phase 5 modules
import sys
import math
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add Model directory to Python path
module_path = Path("../Model").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

from ingestion import ingest_image
from preprocessing import preprocess_image
from segmentation import segment_document
from normalization import normalize_all_characters
from features import (
    sample_virtual_circular,
    get_directional_unit_vectors,
    extract_circular_ddd_features,
    flatten_raster_features,
    extract_features,
    extract_features_batch,
    FeatureExtractionEngine,
)
import config

print("All Phase 5 feature extraction modules loaded successfully!")

---
## Step 2: Extract Normalized 16×16 Characters (Phases 1 to 4)
We run the pipeline on our sample photograph to produce standardized $16 \times 16$ characters.

In [ ]:
# Load test photograph
sample_file = Path("../data/samples/IMG_20261021_212517.jpg")
if not sample_file.exists():
    sample_file = Path("../data/samples/sample_camera_capture_synth.png")

# Run Phases 1 -> 2 -> 3 -> 4
_, ingested, _ = ingest_image(sample_file)
prep = preprocess_image(ingested)
seg = segment_document(prep.binary_mask, prep.gray_image)
norm_result = normalize_all_characters(seg)

print(f"Extracted {norm_result.total_characters} normalized characters of shape 16x16.")

---
## Step 3: Understanding the 8 Compass Directions & Concentric Rings
The 8 directional unit vectors correspond to standard 8-way compass bearings:
- $0^\circ$: East $(1, 0)$
- $45^\circ$: North-East $(0.707, 0.707)$
- $90^\circ$: North $(0, 1)$
- $135^\circ$: North-West $(-0.707, 0.707)$
- $180^\circ$: West $(-1, 0)$
- $225^\circ$: South-West $(-0.707, -0.707)$
- $270^\circ$: South $(0, -1)$
- $315^\circ$: South-East $(0.707, -0.707)$

Along 8 concentric radial rings from $r = 0.47\text{px}$ to $r = 7.03\text{px}$, these rays sweep through the character.

In [ ]:
# Visualize 8-directional ray trajectories overlaid on a sample character
sample_char_img = norm_result.characters[12].normalized_image

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(sample_char_img, cmap="gray", origin="upper", extent=[0, 16, 16, 0])

xc, yc = 7.5, 7.5
dirs = get_directional_unit_vectors(8)
radii = [(m + 0.5) * (7.5 / 8.0) for m in range(8)]

# Draw concentric rings
for r in radii:
    circle = plt.Circle((xc, yc), r, color="cyan", fill=False, linestyle=":", alpha=0.5)
    ax.add_patch(circle)

# Draw sample rays from middle ring
sample_ring_r = radii[3]
for dx, dy in dirs:
    ox = xc + sample_ring_r * dx
    oy = yc + sample_ring_r * dy
    ax.arrow(ox, oy, dx * 3.5, dy * 3.5, head_width=0.4, head_length=0.4, fc="red", ec="red", alpha=0.8)

ax.set_title("Phase 5: Concentric Rings (Cyan) & 8 Compass Rays (Red)", fontsize=11, fontweight="bold")
ax.set_xlim(0, 16)
ax.set_ylim(16, 0)
plt.tight_layout()
plt.show()

---
## Step 4: Extracting the 256-D Feature Vector for a Character
Let us extract the 256-D feature vector for our sample character and inspect its numerical signature.

In [ ]:
# Extract feature vector using Circular DDD
feature_vec = extract_circular_ddd_features(sample_char_img)

print(f"Feature Vector Dimensions : {feature_vec.shape} (Strictly 256 numbers)")
print(f"Data Type                 : {feature_vec.dtype}")
print(f"Feature Value Range       : [{feature_vec.min():.3f}, {feature_vec.max():.3f}]")
print(f"Contains NaNs             : {np.isnan(feature_vec).any()}")
print(f"Contains Infs             : {np.isinf(feature_vec).any()}")

# Plot the 256-feature signature profile
plt.figure(figsize=(14, 3))
plt.plot(feature_vec, color="crimson", linewidth=1.2)
plt.title("256-Dimensional Handcrafted Feature Vector (Numerical Character Signature)", fontsize=11, fontweight="bold")
plt.xlabel("Feature Index (0 to 255)")
plt.ylabel("Normalized Value [0.0, 1.0]")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

---
## Step 5: Feature Discriminability Analysis (Character 'O' vs 'I')
How does the neural network tell letters apart using these 256 numbers?

- A round character like **'O'** has a hollow circular interior: rays cast inward travel through empty space ($W_{\text{span}}$ is high in the center).
- A straight character like **'I'** has a single solid vertical stem: rays cast horizontally hit ink quickly ($W_{\text{dist}}$ is low) and have zero interior loops.

Let us plot their 256-D feature profiles side-by-side to visually confirm how distinct they are!

In [ ]:
# Create synthetic 'O' and 'I' normalized images
char_o = np.zeros((16, 16), dtype=np.float32)
cv2.circle(char_o, (8, 8), 5, 1.0, 2)

char_i = np.zeros((16, 16), dtype=np.float32)
char_i[2:14, 7:9] = 1.0

vec_o = extract_circular_ddd_features(char_o)
vec_i = extract_circular_ddd_features(char_i)

fig, axes = plt.subplots(2, 2, figsize=(14, 6))

axes[0, 0].imshow(char_o, cmap="gray")
axes[0, 0].set_title("Letter 'O' (Hollow Loop)", fontweight="bold")
axes[0, 0].axis("off")

axes[0, 1].plot(vec_o, color="blue", linewidth=1.2)
axes[0, 1].set_title("256-D Feature Signature for 'O'", fontweight="bold")
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].grid(True, linestyle="--", alpha=0.5)

axes[1, 0].imshow(char_i, cmap="gray")
axes[1, 0].set_title("Letter 'I' (Vertical Stem)", fontweight="bold")
axes[1, 0].axis("off")

axes[1, 1].plot(vec_i, color="green", linewidth=1.2)
axes[1, 1].set_title("256-D Feature Signature for 'I'", fontweight="bold")
axes[1, 1].set_ylim(0, 1.05)
axes[1, 1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

euclidean_distance = float(np.linalg.norm(vec_o - vec_i))
print(f"Euclidean Feature Distance between 'O' and 'I' : {euclidean_distance:.2f} (High separation!)")

---
## Step 6: Batch Feature Extraction for the Entire Document
We now transform all 96 extracted characters into a complete dataset matrix of shape $(N, 256)$ ready for the Phase 6 Shallow MLP classifier.

In [ ]:
# Extract features for all characters
feature_engine = FeatureExtractionEngine(mode="circular_ddd")
feature_matrix = feature_engine.extract_batch(norm_result)

print(f"Extracted Feature Matrix Shape : {feature_matrix.shape} (N_characters, N_features)")
print(f"Matrix Data Type              : {feature_matrix.dtype}")
print(f"Global Value Range            : [{feature_matrix.min():.3f}, {feature_matrix.max():.3f}]")
print(f"Total NaN Values               : {np.isnan(feature_matrix).sum()}")
print(f"Total Inf Values               : {np.isinf(feature_matrix).sum()}")
print("Features are 100% verified and ready for Phase 6 neural network classification!")

---
## Quick Professor & Viva Cheat Sheet (Phase 5)

> **Q1: What is Circular DDD feature extraction?**  
> *Answer:* Circular Directional Distance Distribution is a handcrafted computer vision technique that casts rays in 8 directions from concentric circular rings, measuring stroke distance, stroke thickness, loop spaces, and ink density.
>
> **Q2: Why not just use deep convolutional neural networks (CNNs)?**  
> *Answer:* Our project specification strictly prohibits CNNs, deep learning models, and OCR APIs. Handcrafted features with shallow MLPs represent the gold standard of classical, transparent machine learning.
>
> **Q3: What are the W-Set and B-Set metrics?**  
> *Answer:* The W-Set measures white background properties: distance to stroke edge ($W_{\text{dist}}$) and internal loop hollows ($W_{\text{span}}$). The B-Set measures black ink properties: contiguous stroke thickness ($B_{\text{thick}}$) and ink density ($B_{\text{dens}}$).
>
> **Q4: Exactly how is the 256-dimensional vector formed?**  
> *Answer:* We sample across 8 concentric radial rings, cast rays in 8 compass directions at each ring, and compute 4 dual-set metrics per ray: $8 \times 8 \times 4 = 256$ dimensions.
>
> **Q5: What is virtual circular domain wrap-around?**  
> *Answer:* Characters are inscribed inside a circle of radius $R = 7.5$. Coordinates outside this circle are smoothly mapped along polar coordinates ($r \le R_{\max}$, $\theta \pmod{2\pi}$) to prevent boundary cutoff artifacts.